In [1]:
import pandas as pd


In [2]:
from models import Ride, ride_from_row, ride_serializer


In [3]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"

In [4]:
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count','trip_distance','tip_amount','total_amount']
df = pd.read_parquet(url, columns=columns)


In [5]:
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [6]:
from kafka import KafkaProducer

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [7]:


topic_name = 'green-trips-fresh'



In [8]:
# Fill any missing values with 0 so the int() conversion doesn't crash
df = df.fillna(0)

In [9]:
import time

# 1. Clean the missing data so the int() conversion doesn't crash halfway through
df = df.fillna(0)

print(f"Sending {len(df)} rows to {topic_name}...")

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    # 2. Just send it. No printing, no sleeping!
    producer.send(topic_name, value=ride)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

Sending 49416 rows to green-trips-fresh...
took 10.17 seconds


In [5]:
import pandas as pd

# 1. Ensure the column is a proper datetime object
df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'], errors='coerce')

# 2. Drop the dirty data that was crashing Flink
clean_df = df.dropna(subset=['lpep_pickup_datetime'])

# 3. Simulate the Flink Tumbling Window (Group by 5 minutes AND Location ID)
windowed_counts = clean_df.groupby([
    pd.Grouper(key='lpep_pickup_datetime', freq='5min'), 
    'PULocationID'
]).size().reset_index(name='num_trips')

# 4. Find the top 3 busiest 5-minute windows
top_locations = windowed_counts.sort_values(by='num_trips', ascending=False).head(3)

print("Top 3 locations by 5-minute tumbling window:")
print(top_locations[['PULocationID', 'num_trips']])

Top 3 locations by 5-minute tumbling window:
       PULocationID  num_trips
23500            74         15
21829            74         14
15690            74         13


In [6]:
import pandas as pd

# 1. Clean the timestamps and sort chronologically by location
df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'], errors='coerce')
clean_df = df.dropna(subset=['lpep_pickup_datetime']).sort_values(['PULocationID', 'lpep_pickup_datetime'])

# 2. Find the time gap between each consecutive pickup at the same location
clean_df['time_gap'] = clean_df.groupby('PULocationID')['lpep_pickup_datetime'].diff()

# 3. If the gap is greater than 5 minutes (or it's the very first trip), mark it as a NEW session
clean_df['is_new_session'] = (clean_df['time_gap'] > pd.Timedelta(minutes=5)) | clean_df['time_gap'].isna()

# 4. Group the data by location AND these unique sessions, then count the rows
clean_df['session_id'] = clean_df.groupby('PULocationID')['is_new_session'].cumsum()
session_counts = clean_df.groupby(['PULocationID', 'session_id']).size()

# 5. Print the absolute maximum streak!
print(f"Max trips in a single session: {session_counts.max()}")

Max trips in a single session: 82


In [8]:
# 1. We must set the timestamp as the index so Pandas can chop it into time blocks
hourly_df = clean_df.set_index('lpep_pickup_datetime')

# 2. Resample into 1-hour tumbling windows and sum the tips
hourly_tips = hourly_df.resample('1h')['tip_amount'].sum().reset_index()

# 3. Find the single hour with the highest total tip amount
best_hour = hourly_tips.loc[hourly_tips['tip_amount'].idxmax()]

print("The hour with the highest total tips was:")
print(best_hour)

The hour with the highest total tips was:
lpep_pickup_datetime    2025-10-16 18:00:00
tip_amount                           524.96
Name: 524, dtype: object
